# Compare two runs: Adaptive (HS) vs Adaptive (Covariance)

Provide **two paths** to `timeseries.csv` (one HS, one Covar).  
Optionally provide an **exact** `timeseries.csv` (or an exact column inside the CSV).


In [ ]:
import numpy as np
import qutip as qt

def heisenberg_XYZ_open(N: int, Jx: float, Jy: float, Jz: float) -> qt.Qobj:
    """Open-boundary XYZ chain: H = sum_{i=0}^{N-2} (Jx X_i X_{i+1} + Jy Y_i Y_{i+1} + Jz Z_i Z_{i+1})."""
    sx, sy, sz = qt.sigmax(), qt.sigmay(), qt.sigmaz()
    si = qt.qeye(2)

    def op_on_sites(op_i, i, op_j, j):
        ops = [si] * N
        ops[i] = op_i
        ops[j] = op_j
        return qt.tensor(ops)

    H = 0
    for i in range(N - 1):  # open BCs
        H += Jx * op_on_sites(sx, i, sx, i + 1)
        H += Jy * op_on_sites(sy, i, sy, i + 1)
        H += Jz * op_on_sites(sz, i, sz, i + 1)
    return qt.Qobj(H)

def sigma0_z_polarized(N: int) -> qt.Qobj:
    """Product density matrix |0><0|^{⊗N} (max +Z polarization, no x/y, no entanglement)."""
    ket0 = qt.basis(2, 0)          # |0> : +1 eigenstate of sigmaz
    rho0 = ket0 * ket0.dag()       # |0><0|
    return qt.tensor([rho0] * N)

def Sz0(N: int, site: int = 0) -> qt.Qobj:
    """Single-site observable σ^z_site (default site 0)."""
    sz = qt.sigmaz()
    si = qt.qeye(2)
    ops = [si] * N
    ops[site] = sz
    return qt.tensor(ops)

# --- Example usage ---
N = 10
Jx, Jy, Jz = 1.0, -1, 1e-1   # inputs
H = heisenberg_XYZ_open(N, Jx, Jy, Jz)

rho0 = sigma0_z_polarized(N)
O = Sz0(N, site=0)

# time grid, ħ=1 (QuTiP uses ħ=1 units by default)
tlist = np.linspace(0.0, 1.0, 100)

# Schrödinger picture: <O>(t) = Tr[ O ρ(t) ]
res = qt.mesolve(H, rho0, tlist, c_ops=[], e_ops=[O])
exp_Sz0 = np.array(res.expect[0])/2

print("⟨Sz0⟩(t=0) =", exp_Sz0[0])


In [7]:
# --- Inputs: set your two paths here ---
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

folder_path_ = "home/tomas/phoenix/data/"
_file_path = "/timeseries.csv"
# Required
path_hs    = Path(
            folder_path_ +
            "20260205_153944__XY__N10__ell5__m2__eps1e-10__ptail2__dt4.000e-03__tmax0.4__seedZ0__sigma_HS" 
            + _file_path
            ).expanduser()
path_covar = Path(
            folder_path_ +
            "/20260205_151128__XY__N10__ell5__m2__eps1e-10__ptail2__dt1.000e-02__tmax1__seedZ0__sigmasigmaZ_global" 
            + _file_path
            ).expanduser()


# 
# If you want to force a specific observable column name, set it here. Otherwise it's auto-detected.
observable_col_preference = None

# Plot
PLOT_DPI = 140


In [8]:
# --- Helpers ---
def detect_time_col(df: pd.DataFrame) -> str:
    for cand in ["t","time"]:
        if cand in df.columns:
            return cand
    # fallback: first numeric col
    nums = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not nums:
        raise ValueError("No numeric columns found (cannot detect time column).")
    return nums[0]

def detect_observable_col(df: pd.DataFrame, preferred: str | None = None) -> str:
    if preferred is not None and preferred in df.columns:
        return preferred

    # common names in your runs
    cands = [
        "O_{\ell, m, \epsilon}",
        "O_{\ell,m,\epsilon}",
        "O_lm_eps",
        "obs",
        "ev",
        "value",
    ]
    for c in cands:
        if c in df.columns and pd.api.types.is_numeric_dtype(df[c]):
            return c

    tcol = detect_time_col(df)
    nums = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != tcol]
    if not nums:
        raise ValueError("No numeric observable column found.")
    return nums[0]

def standardize(df: pd.DataFrame, label: str) -> pd.DataFrame:
    df = df.copy()
    tcol = detect_time_col(df)
    ocol = detect_observable_col(df, observable_col_preference)
    df = df.rename(columns={tcol:"t", ocol:"y"})
    df["t"] = df["t"].astype(float)
    df["y"] = df["y"].astype(float)
    df["label"] = label
    return df.sort_values("t").reset_index(drop=True)

def parse_meta_from_path(p: Path) -> dict:
    # best-effort parse from parent folder name
    name = p.parent.name
    meta = {}
    parts = name.split("__")
    # model often is the first token after timestamp
    if parts and re.match(r"^\d{8}_\d{6}$", parts[0]):
        parts = parts[1:]
    if parts:
        meta["model"] = parts[0]
    for token in parts[1:]:
        m = re.match(r"^(N|ell|m)(\d+)$", token)
        if m:
            meta[m.group(1)] = int(m.group(2))
        m = re.match(r"^eps(.+)$", token)
        if m:
            try:
                meta["eps"] = float(m.group(1))
            except Exception:
                meta["eps"] = m.group(1)
    return meta

def nice_title(meta: dict) -> str:
    model = meta.get("model","?")
    N     = meta.get("N","?")
    ell   = meta.get("ell","?")
    m     = meta.get("m","?")
    eps   = meta.get("eps","?")
    # eps formatting
    if isinstance(eps, float):
        eps_str = f"{eps:g}"
    else:
        eps_str = str(eps)
    return f"{model} (N={N}) | ℓ={ell}, m={m}, ε={eps_str}"


<>:18: SyntaxWarning: invalid escape sequence '\e'
<>:19: SyntaxWarning: invalid escape sequence '\e'
<>:18: SyntaxWarning: invalid escape sequence '\e'
<>:19: SyntaxWarning: invalid escape sequence '\e'
/tmp/ipykernel_76277/2215471689.py:18: SyntaxWarning: invalid escape sequence '\e'
  "O_{\ell, m, \epsilon}",
/tmp/ipykernel_76277/2215471689.py:19: SyntaxWarning: invalid escape sequence '\e'
  "O_{\ell,m,\epsilon}",


In [9]:

# --- Load (simple, obs-based) ---
assert path_hs.exists(), f"HS timeseries.csv not found: {path_hs}"
assert path_covar.exists(), f"Covar timeseries.csv not found: {path_covar}"

df_hs  = pd.read_csv(path_hs)
df_cov = pd.read_csv(path_covar)

# Expect columns: t, obs, tail_frac, rebuilt
for df, name in [(df_hs,"HS"), (df_cov,"Covar")]:
    assert "t" in df.columns and "obs" in df.columns, f"{name} CSV must contain columns t, obs"

df_hs  = df_hs[["t","obs"]].rename(columns={"obs":"y_hs"})
df_cov = df_cov[["t","obs"]].rename(columns={"obs":"y_cov"})

# Optional exact (QuTiP or embedded)
df_exact = None
if path_exact is not None:
    path_exact = Path(path_exact).expanduser()
    assert path_exact.exists(), f"Exact timeseries.csv not found: {path_exact}"
    df_e = pd.read_csv(path_exact)
    assert "t" in df_e.columns and "obs" in df_e.columns
    df_exact = df_e[["t","obs"]].rename(columns={"obs":"y_exact"})
else:
    # try embedded exact column (e.g. QuTiP runs)
    for df_raw in [pd.read_csv(path_hs), pd.read_csv(path_covar)]:
        for cand in ["ev_exact","exact","y_exact"]:
            if cand in df_raw.columns:
                df_exact = df_raw[["t",cand]].rename(columns={cand:"y_exact"})
                print(f"Built Exact reference from embedded column: {cand}")
                break
        if df_exact is not None:
            break

# Align on common t
aligned = df_hs.merge(df_cov, on="t", how="inner")
if df_exact is not None:
    aligned = aligned.merge(df_exact, on="t", how="inner")

aligned.head()


AssertionError: HS timeseries.csv not found: home/tomas/phoenix/data/20260205_153944__XY__N10__ell5__m2__eps1e-10__ptail2__dt4.000e-03__tmax0.4__seedZ0__sigma_HS/timeseries.csv

In [ ]:
# --- Plot(s) ---
plt.rcParams.update({
    "figure.dpi": PLOT_DPI,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

meta = parse_meta_from_path(path_hs)
supt = nice_title(meta)

# 1) Observable vs time
fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(aligned["t"], aligned["y_hs"],  marker="o", markersize=3, lw=1.8, label="Adaptive (HS)")
ax.plot(aligned["t"], aligned["y_cov"], marker="o", markersize=3, lw=1.8, label="Adaptive (Covariance)")

if "y_exact" in aligned.columns:
    ax.plot(aligned["t"], aligned["y_exact"], marker="o", markersize=3, lw=2.2, ls="--", label="Exact")

ax.set_xlabel("t")
ax.set_ylabel("obs")
ax.set_title("Observable vs time")
ax.legend(loc="best")
fig.suptitle(supt, y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# 2) |HS - Covar|
fig, ax = plt.subplots(figsize=(9, 4.4))
diff = np.abs(aligned["y_hs"] - aligned["y_cov"])
ax.plot(aligned["t"], diff, marker="o", markersize=3, lw=1.8, label="|HS − Covar|")
ax.set_xlabel("t")
ax.set_ylabel("abs diff")
ax.set_yscale("log")
ax.set_title("Geometry-induced difference (log scale)")
ax.legend(loc="best")
fig.suptitle(supt, y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

# 3) Errors vs exact (if present)
if "y_exact" in aligned.columns:
    fig, ax = plt.subplots(figsize=(9, 4.4))
    err_hs  = np.abs(aligned["y_hs"]  - aligned["y_exact"])
    err_cov = np.abs(aligned["y_cov"] - aligned["y_exact"])
    ax.plot(aligned["t"], err_hs,  marker="o", markersize=3, lw=1.8, label="|HS − Exact|")
    ax.plot(aligned["t"], err_cov, marker="o", markersize=3, lw=1.8, label="|Covar − Exact|")
    ax.set_xlabel("t")
    ax.set_ylabel("abs err")
    ax.set_yscale("log")
    ax.set_title("Absolute error vs Exact (log scale)")
    ax.legend(loc="best")
    fig.suptitle(supt, y=1.02, fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print("No exact reference found/provided → skipping error-vs-exact plot.")
